In [12]:
from pathlib import Path
import torch

from segmentation.config import DATA_CFG, SEED
from segmentation.data import resolve_paths, split_pairs_by_group
from segmentation.splits import (
    build_or_load_training_split_manifest,
    split_from_manifest,
)

split_manifest = build_or_load_training_split_manifest(force_rebuild=False)
split = split_from_manifest(split_manifest, split_key="train_notebook_split")


data_cfg = dict(DATA_CFG)
seed = SEED  # use the same seed you pass to prepare_data_pipeline(...)

data_cfg["x_train_files"] = split["x_train"]
data_cfg["y_train_files"] = split["y_train"]
data_cfg["x_val_files"] = split["x_val"]
data_cfg["y_val_files"] = split["y_val"]
data_cfg["x_test_files"] = split["x_test"]
data_cfg["y_test_files"] = split["y_test"]
data_cfg["group_split_enabled"] = False



x_all = resolve_paths(data_cfg["x_train_files"])
y_all = resolve_paths(data_cfg["y_train_files"])

x_val_exp = resolve_paths(data_cfg["x_val_files"]) if data_cfg["x_val_files"] else []
y_val_exp = resolve_paths(data_cfg["y_val_files"]) if data_cfg["y_val_files"] else []
x_test_exp = resolve_paths(data_cfg["x_test_files"]) if data_cfg.get("x_test_files") else []
y_test_exp = resolve_paths(data_cfg["y_test_files"]) if data_cfg.get("y_test_files") else []

if x_val_exp or y_val_exp or x_test_exp or y_test_exp:
    # Mirrors current prepare_data_pipeline behavior exactly.
    x_train_files, y_train_files = x_all, y_all
    x_val_files, y_val_files = x_val_exp, y_val_exp
    x_test_files, y_test_files = x_test_exp, y_test_exp
else:
    if data_cfg.get("group_split_enabled", True):
        x_train_files, y_train_files, x_val_files, y_val_files, x_test_files, y_test_files = split_pairs_by_group(
            x_all,
            y_all,
            root=data_cfg.get("group_split_root", "tensors"),
            ratios=tuple(data_cfg.get("split_ratios", (0.8, 0.1, 0.1))),
            seed=data_cfg.get("split_seed", seed),
        )
    else:
        n_total = len(x_all)
        n_val = max(1, int(round(n_total * float(data_cfg.get("val_fraction", 0.1)))))
        n_val = min(n_val, n_total - 1)

        g = torch.Generator().manual_seed(int(seed))
        perm = torch.randperm(n_total, generator=g).tolist()
        val_idx = set(perm[:n_val])

        x_train_files, y_train_files, x_val_files, y_val_files = [], [], [], []
        for i, (x, y) in enumerate(zip(x_all, y_all)):
            if i in val_idx:
                x_val_files.append(x)
                y_val_files.append(y)
            else:
                x_train_files.append(x)
                y_train_files.append(y)
        x_test_files, y_test_files = [], []

splits = {
    "train": x_train_files,
    "val": x_val_files,
    "test": x_test_files,
}

for name, files in splits.items():
    print(f"\n{name}: {len(files)} files")
    for p in files[:10]:
        print(" ", p)
    Path(f"{name}_rawavg_files.txt").write_text("\n".join(files) + "\n")

print("\noverlap checks")
print("train ∩ val :", len(set(x_train_files) & set(x_val_files)))
print("train ∩ test:", len(set(x_train_files) & set(x_test_files)))
print("val ∩ test  :", len(set(x_val_files) & set(x_test_files)))



train: 3408 files
  /home/rph/convolutional_ar/segmentation/tensors/convolutional_ar/segmentation/hcp_filt/671855/T1w/rawavg.pt
  /home/rph/convolutional_ar/segmentation/tensors/convolutional_ar/segmentation/hcp_filt/237334/T1w/rawavg.pt
  /home/rph/convolutional_ar/segmentation/tensors/convolutional_ar/segmentation/hcp_filt/255740/T1w/rawavg.pt
  /home/rph/convolutional_ar/segmentation/tensors/convolutional_ar/segmentation/hcp_filt/990366/T1w/rawavg.pt
  /home/rph/convolutional_ar/segmentation/tensors/convolutional_ar/segmentation/hcp_filt/125525/T1w/rawavg.pt
  /home/rph/convolutional_ar/segmentation/tensors/convolutional_ar/segmentation/hcp_filt/856766/T1w/rawavg.pt
  /home/rph/convolutional_ar/segmentation/tensors/convolutional_ar/segmentation/hcp_filt/303119/T1w/rawavg.pt
  /home/rph/convolutional_ar/segmentation/tensors/convolutional_ar/segmentation/hcp_filt/656253/T1w/rawavg.pt
  /home/rph/convolutional_ar/segmentation/tensors/convolutional_ar/segmentation/hcp_filt/922854/T1w/r